# Risk Management System - Quick Start Guide

This notebook demonstrates the basic usage of the RMS system.

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import DataLoader
from risk_engine import RiskEngine
from portfolio import Portfolio
from ml_model import RiskPredictor

%matplotlib inline
sns.set_style('whitegrid')

## 1. Load Market Data

In [ ]:
loader = DataLoader(data_dir='../data')

tickers = ['2222.SR', '1120.SR', '2010.SR']
prices = loader.get_close_prices(tickers, period='1y')
returns = loader.get_returns(tickers, period='1y')
volume = loader.get_volume_data(tickers, period='1y')

print(f"Data shape: {prices.shape}")
print(f"Date range: {prices.index[0]} to {prices.index[-1]}")
prices.head()

## 2. Create Portfolio

In [ ]:
portfolio = Portfolio(
    name="Saudi Portfolio",
    initial_value=1000000,
    tickers=tickers,
    weights=np.array([0.4, 0.3, 0.3])
)

latest_prices = prices.iloc[-1]
portfolio.update_prices(latest_prices)

portfolio.get_portfolio_summary()

## 3. Calculate Risk Metrics

In [ ]:
risk_engine = RiskEngine(returns, confidence_levels=[0.95, 0.99])

returns_stats = risk_engine.calculate_returns_stats()
print("Annualized Returns:")
print(returns_stats['annualized_return'])
print("\nAnnualized Volatility:")
print(returns_stats['annualized_volatility'])

## 4. Value at Risk (VaR)

In [ ]:
weights = portfolio.weights
portfolio_value = portfolio.current_value

var_param = risk_engine.calculate_parametric_var(portfolio_value, weights)
var_hist = risk_engine.calculate_historical_var(portfolio_value, weights)
cvar = risk_engine.calculate_cvar(portfolio_value, weights)

print("VaR at 95% confidence:")
print(f"  Parametric: ${var_param[0.95]:,.2f}")
print(f"  Historical: ${var_hist[0.95]:,.2f}")
print(f"  CVaR: ${cvar[0.95]:,.2f}")

## 5. Correlation Analysis

In [ ]:
correlation_matrix = risk_engine.calculate_correlation_matrix()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Asset Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Monte Carlo Simulation

In [ ]:
mc_results = risk_engine.monte_carlo_simulation(
    portfolio_value=portfolio_value,
    weights=weights,
    num_simulations=5000,
    time_horizon=252
)

plt.figure(figsize=(12, 6))
for i in range(min(100, 5000)):
    plt.plot(mc_results['simulated_paths'][i] * portfolio_value, alpha=0.1, linewidth=0.5)

mean_path = mc_results['simulated_paths'].mean(axis=0) * portfolio_value
plt.plot(mean_path, 'r-', linewidth=2, label='Mean Path')
plt.axhline(y=portfolio_value, color='black', linestyle='--', label='Initial Value')

plt.xlabel('Days')
plt.ylabel('Portfolio Value ($)')
plt.title('Monte Carlo Simulation - 5000 Paths')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean Final Value: ${mc_results['mean_final_value']:,.2f}")
print(f"5th Percentile: ${mc_results['percentiles']['5th']:,.2f}")
print(f"95th Percentile: ${mc_results['percentiles']['95th']:,.2f}")

## 7. Machine Learning Risk Prediction

In [ ]:
predictor = RiskPredictor(model_type='classification')

features = predictor.prepare_features(returns, prices, volume)
target = predictor.create_target_classification(returns, weights)

train_results = predictor.train_model(features, target, test_size=0.2)

print(f"Training Accuracy: {train_results['train_accuracy']:.4f}")
print(f"Test Accuracy: {train_results['test_accuracy']:.4f}")

feature_importance = train_results['feature_importance'].head(10)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Top 10 Feature Importances')
plt.tight_layout()
plt.show()

## 8. Current Risk Prediction

In [ ]:
latest_features = features.iloc[-1:]
risk_level = predictor.predict_risk_level(latest_features)

print(f"Current Predicted Risk Level: {risk_level[0]}")

risk_colors = {'LOW': '🟢', 'MEDIUM': '🟡', 'HIGH': '🔴'}
print(f"{risk_colors.get(risk_level[0], '⚪')} Risk Level: {risk_level[0]}")

## 9. Comprehensive Risk Report

In [ ]:
risk_summary = risk_engine.get_risk_summary(portfolio_value, weights)
risk_summary